# Statistical Analysis

## load Data

In [ ]:
import json
from pathlib import Path
import pandas as pd

BASE_DIR = Path("../../data/01_raw")

metadata = []

for json_path in BASE_DIR.rglob("*.json"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

            data["_source_folder"] = json_path.parent.name
            data["_file_name"] = json_path.name

            metadata.append(data)

    except (json.JSONDecodeError, IOError) as e:
        print(f"Fehler beim Lesen von {json_path}: {e}")

df = pd.DataFrame(metadata)
print(df)


                                    Titel_und_Signatur Prüfnummer   Prüfdatum  \
0         R 9346-I/52\nMenschliche Hyänen\n1920 - 1945        267   19.8.1920   
1      R 9346-I/101\nBobbys unglücklichster Tag.\n1920        533   8.10.1920   
2    R 9346-I/108\nDas Geheimnis der Spielhölle von...        564  16.10.1920   
3         R 9346-I/80\nEin schwaches Weib\n1920 - 1945        430   16.9.1920   
4               R 9346-I/21\nDoktor Klaus\n1920 - 1945         93    8.7.1920   
..                                                 ...        ...         ...   
117  R 9346-I/40\nDer Sklavenhalter von Kansas-City...        209   10.8.1920   
118           R 9346-I/62\nSatan Diktator\n1920 - 1945        345    1.9.1920   
119  R 9346-I/36\nVerborgene Wunder unserer Gewässe...        181    2.8.1920   
120          R 9346-I/86\nDas rote Plakat\n1920 - 1945        444    5.9.1920   
121            R 9346-I/90\nDer Spreewald\n1920 - 1945        465   20.9.1920   

                      Antra

## Reconstruction lost Documents

In [14]:
# 1. Datenbereinigung & Extraktion
# Extrahiert die erste 4-stellige Zahl aus dem Prüfdatum als Jahr (robuster als pd.to_datetime bei unvollständigen Daten)
df['Jahr'] = df['Prüfdatum'].astype(str).str.extract(r'(\d{4})')[0].astype(float).astype('Int64')

# Bereinigt die Prüfnummer (extrahiert nur die Ziffern, falls Suffixe wie 'a' oder 'b' existieren)
df['Prüfnummer_clean'] = pd.to_numeric(df['Prüfnummer'].astype(str).str.extract(r'(\d+)')[0], errors='coerce').astype('Int64')

# 2. Aggregation pro Jahr
summary = df.groupby('Jahr').agg(
    Eintraege_im_DF=('Prüfnummer_clean', 'count'),
    Anfangs_Prüfnummer=('Prüfnummer_clean', 'min'),
    End_Prüfnummer=('Prüfnummer_clean', 'max')
).reset_index()

# 3. Berechnung der Differenzen
# Die theoretische Gesamtzahl basiert auf der Annahme einer lückenlosen Sequenz: (Max - Min + 1)
summary['Theoretische_Dokumente'] = summary['End_Prüfnummer'] - summary['Anfangs_Prüfnummer'] + 1
summary['Differenz_Fehlend'] = summary['Theoretische_Dokumente'] - summary['Eintraege_im_DF']

# Spaltenreihenfolge für bessere Lesbarkeit anpassen
summary = summary[['Jahr', 'Anfangs_Prüfnummer', 'End_Prüfnummer', 'Theoretische_Dokumente', 'Eintraege_im_DF', 'Differenz_Fehlend']]

print(summary.to_string(index=False))

 Jahr  Anfangs_Prüfnummer  End_Prüfnummer  Theoretische_Dokumente  Eintraege_im_DF  Differenz_Fehlend
 1920                   2             669                     668              117                551
 1921                  22             660                     639                5                634


In [13]:
# extract year from date
df['Jahr'] = df['Prüfdatum'].astype(str).str.extract(r'(\d{4})')[0].astype(float).astype('Int64')

summary = df.groupby('Jahr').agg(
    Dokumente_im_DF=('Prüfnummer', 'count'),
    Anfangs_Prüfnummer=('Prüfnummer', 'min'),
    End_Prüfnummer=('Prüfnummer', 'max')
).reset_index()

summary['Theoretische_Dokumente'] = summary['End_Prüfnummer'] - summary['Anfangs_Prüfnummer'] + 1

summary['Differenz_Fehlend'] = summary['Theoretische_Dokumente'] - summary['Dokumente_im_DF']

summary = summary[['Jahr', 'Anfangs_Prüfnummer', 'End_Prüfnummer', 'Theoretische_Dokumente', 'Dokumente_im_DF', 'Differenz_Fehlend']]

print(summary.to_string(index=False))

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [ ]:
import json
from pathlib import Path
import pandas as pd

# 1. Hauptverzeichnis definieren, in dem die Dokumentenordner liegen
BASE_DIR = Path("../../data/01_raw")

all_metadata = []

# 2. Rekursiv alle JSON-Dateien in den Unterordnern suchen
# rglob("*.json") findet jede JSON-Datei, egal wie tief die Ordnerstruktur ist
for json_path in BASE_DIR.rglob("*.json"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            # Best Practice: Metadaten um den Ordnernamen erweitern, 
            # falls die JSON selbst keine eindeutige ID enthält.
            data["_source_folder"] = json_path.parent.name
            data["_file_name"] = json_path.name
            
            all_metadata.append(data)
            
    except (json.JSONDecodeError, IOError) as e:
        print(f"Fehler beim Lesen von {json_path}: {e}")

# 3. Daten in ein Pandas DataFrame laden
# Falls deine JSONs flach sind (keine verschachtelten Dicts/Listen):
df = pd.DataFrame(all_metadata)
print(df)

                                    Titel_und_Signatur Prüfnummer   Prüfdatum  \
0         R 9346-I/52\nMenschliche Hyänen\n1920 - 1945        267   19.8.1920   
1      R 9346-I/101\nBobbys unglücklichster Tag.\n1920        533   8.10.1920   
2    R 9346-I/108\nDas Geheimnis der Spielhölle von...        564  16.10.1920   
3         R 9346-I/80\nEin schwaches Weib\n1920 - 1945        430   16.9.1920   
4               R 9346-I/21\nDoktor Klaus\n1920 - 1945         93    8.7.1920   
..                                                 ...        ...         ...   
117  R 9346-I/40\nDer Sklavenhalter von Kansas-City...        209   10.8.1920   
118           R 9346-I/62\nSatan Diktator\n1920 - 1945        345    1.9.1920   
119  R 9346-I/36\nVerborgene Wunder unserer Gewässe...        181    2.8.1920   
120          R 9346-I/86\nDas rote Plakat\n1920 - 1945        444    5.9.1920   
121            R 9346-I/90\nDer Spreewald\n1920 - 1945        465   20.9.1920   

                      Antra

In [ ]:
import json
from pathlib import Path
import pandas as pd

# 1. Hauptverzeichnis definieren, in dem die Dokumentenordner liegen
BASE_DIR = Path("/pfad/zu/deinen/historischen_dokumenten")

all_metadata = []

# 2. Rekursiv alle JSON-Dateien in den Unterordnern suchen
# rglob("*.json") findet jede JSON-Datei, egal wie tief die Ordnerstruktur ist
for json_path in BASE_DIR.rglob("*.json"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            # Best Practice: Metadaten um den Ordnernamen erweitern, 
            # falls die JSON selbst keine eindeutige ID enthält.
            data["_source_folder"] = json_path.parent.name
            data["_file_name"] = json_path.name
            
            all_metadata.append(data)
            
    except (json.JSONDecodeError, IOError) as e:
        # Robustheit: Fehlerhafte JSONs dokumentieren, statt das Skript abbrechen zu lassen
        print(f"Fehler beim Lesen von {json_path}: {e}")

# 3. Daten in ein Pandas DataFrame laden
# Falls deine JSONs flach sind (keine verschachtelten Dicts/Listen):
df = pd.DataFrame(all_metadata)

# HINWEIS: Falls deine JSONs tief verschachtelt sind, nutze stattdessen:
# df = pd.json_normalize(all_metadata)


# ==============================================================================
# 4. ANALYSE-BEISPIELE (ZÄHLEN)
# ==============================================================================

print("--- Struktur der Daten ---")
print(df.info())

# Beispiel A: Häufigkeit eines bestimmten Feldes (z.B. 'autor' oder 'epoche') zählen
# value_counts() liefert dir die absolute Häufigkeit sortiert im absteigender Reihenfolge
if "epoche" in df.columns:
    print("\n--- Dokumente pro Epoche ---")
    print(df["epoche"].value_counts())

# Beispiel B: Relative Häufigkeiten (Prozentualer Anteil)
if "jahr" in df.columns:
    print("\n--- Verteilung der Jahre (Top 5 in %) ---")
    print(df["jahr"].value_counts(normalize=True).head(5) * 100)

# Beispiel C: Fehlende Metadaten zählen (Qualitätskontrolle der Dokumente)
print("\n--- Fehlende Werte pro Metadaten-Feld ---")
print(df.isnull().sum())